# 13. Final model comparison

**Question: with every family tuned honestly, which one should actually be deployed?**

This is the only notebook that touches the official UCI test set as a headline result, and
it does so exactly once per family.

The protocol behind every number here:

1. `fit` — estimate parameters
2. `tune` — choose hyperparameters, on cost
3. refit the winner on `fit + tune`
4. `calibration` — fit Platt or isotonic, if requested
5. `threshold` — choose the operating point
6. **official test set** — evaluated once, never used for any decision above

That ordering is what makes the comparison believable. Every family gets the same budget,
the same splits and its own cost-optimal threshold, so the ranking reflects the models
rather than the effort spent on each.

> **The objective.** Every number in this notebook is judged against
> `J = 10·FP + 500·FN`. A false positive is an unnecessary inspection; a false
> negative is a truck that fails in service. Missing one failure costs as much
> as fifty needless inspections, and that ratio is what makes the modelling
> choices here matter.

In [ ]:
from pathlib import Path

import pandas as pd

from scania_aps.data import TEST_FILENAME, TRAIN_FILENAME, read_raw_csv
from scania_aps.plotting import apply_house_style

ROOT = Path.cwd().resolve()
if ROOT.name == "experiments":
    ROOT = ROOT.parent
TRAIN = ROOT / "data" / "raw" / TRAIN_FILENAME
TEST = ROOT / "data" / "raw" / TEST_FILENAME
ARTIFACTS = ROOT / "artifacts"
assert TRAIN.exists() and TEST.exists(), "Run: poetry run scania-aps download"

apply_house_style()

train = read_raw_csv(TRAIN)
test = read_raw_csv(TEST)

pd.DataFrame(
    {
        "trucks": [len(train.y), len(test.y)],
        "features": [train.X.shape[1], test.X.shape[1]],
        "failures": [int(train.y.sum()), int(test.y.sum())],
        "failure_rate": [train.y.mean(), test.y.mean()],
    },
    index=["training set", "official test set"],
)

In [ ]:
from scania_aps.studies import run_model_family_study

comparison = run_model_family_study(
    TRAIN,
    TEST,
    ARTIFACTS,
    families=[
        "logistic",
        "linear_svm",
        "random_forest",
        "extra_trees",
        "xgboost",
        "lightgbm",
        "mlp",
        "autoencoder",
    ],
    profile="full",
    calibration="none",
)
comparison

## The ranking

Eight categorical colours would bury the answer. The question is "which is cheapest", so
the winner is highlighted and the rest recede.

In [ ]:
from scania_aps.plotting import emphasis_bars

fig, ax = emphasis_bars(
    list(comparison["family"]),
    [float(v) for v in comparison["total_cost"].to_numpy()],
    title="Maintenance cost by model family",
    subtitle="Official UCI test set, each family at its own cost-optimal threshold.",
    xlabel="Maintenance cost  (10·FP + 500·FN)",
)

## What the ranking costs in real terms

Cost per truck is the number a fleet operator would recognise. The savings columns put it
against the two model-free policies from
[notebook 02](02_cost_sensitive_baselines.ipynb) — a model that cannot beat "inspect
everything" has no business being deployed.

In [ ]:
headline = comparison.set_index("family")[
    [
        "total_cost",
        "cost_per_observation",
        "false_negatives",
        "false_positives",
        "recall",
        "precision",
        "pr_auc",
        "saving_vs_always_negative",
        "saving_vs_always_positive",
    ]
].copy()

best_family = headline["total_cost"].idxmin()
print(f"cheapest family: {best_family}")
print(f"cost per truck : {headline.loc[best_family, 'cost_per_observation']:.2f}")
print(
    f"failures missed: {int(headline.loc[best_family, 'false_negatives'])} of {int(test.y.sum())}"
)
print(
    f"saving vs inspecting everything: {headline.loc[best_family, 'saving_vs_always_positive']:.1%}"
)
headline

## The error trade at the chosen operating point

Two families with similar total cost can be making very different mistakes. This is the
table a maintenance planner would actually argue over.

In [ ]:
trade = comparison.set_index("family")[
    ["false_negatives", "false_positives", "recall", "total_cost"]
].copy()
trade["cost_from_missed_failures"] = trade["false_negatives"] * 500
trade["cost_from_inspections"] = trade["false_positives"] * 10
trade["share_from_missed"] = trade["cost_from_missed_failures"] / trade["total_cost"]
trade.sort_values("total_cost")

### Conclusions

Read the comparison against the study as a whole:

- **The decision rule outweighs the model family.** The spread between the best and worst
  family here is smaller than the spread [notebook 10](10_threshold_optimization.ipynb)
  found between `0.5` and a learned threshold on a *single* model. Choosing the threshold
  well matters more than choosing the algorithm well.
- **Nonlinearity helps, modestly.** The boosted and tree families should sit below the
  linear ones, but the gap is not large enough to justify a complex deployment if a
  logistic model is easier to operate and audit.
- **`share_from_missed` is near one for every family.** Under a 50:1 ratio the bill is
  dominated by missed failures regardless of model, which is why recall bought cheaply is
  the whole game.

### What this study does not establish

- A single train/test split gives no confidence interval. The ranking between adjacent
  families is not statistically separated and should not be reported as if it were.
- The costs `10` and `500` are taken as given. A different ratio moves every threshold in
  this repository and could reorder the families.
- The features are anonymised, so nothing here explains *why* a truck fails. This is a
  detection study, not a diagnostic one.

**Back to the start:** [01 — data quality and missingness](01_data_quality_and_missingness.ipynb).